# Data acquisition

**Goal:** go from "what exists in the archive?" to a clean, reproducible set of files on disk, already split by night.

Read `docs/data_acquisition.md` alongside this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.query_ztf_metadata import query_ztf_metadata
from asthunt.acquire.select_night_exposures import select_night_exposures
from asthunt.acquire.download_ztf_images import download_ztf_images
from asthunt.acquire.download_ztf_reference_image import download_ztf_reference_image

paths = load_config("project")["paths"]
acquire = load_config("acquire")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


raw_dir = paths["raw"] / "ztf"

## 1. Ask the archive what exists
One web request returns one row per exposure. It's cached in `data/raw/ztf/metadata__000618.csv`,
so the next run reads the file instead of the internet, and we keep a record of exactly what we saw.

In [ ]:
metadata = load_unit_metadata(unit_of(None), paths["raw"])
print(len(metadata), "exposures")
metadata[["obsdate", "filtercode", "seeing", "maglimit", "infobits", "moonillf"]].head()

## 2. Explore before choosing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metadata["filtercode"].value_counts().plot.bar(ax=axes[0], title="exposures per filter")
axes[1].hist(metadata["seeing"], bins=50); axes[1].set_title("seeing (FWHM, pixels)")
axes[2].hist(metadata["maglimit"], bins=50); axes[2].set_title("limiting magnitude")
plt.show()

print("fraction with quality flags (infobits != 0):", round((metadata["infobits"] != 0).mean(), 3))

## 3. Which nights can we link?
Linking needs **several exposures of the same field on the same night**.

In [ ]:
good_r = metadata[(metadata["filtercode"] == acquire["filter"]) & (metadata["infobits"] == 0)]
nights = good_r.groupby("night").agg(
    frames=("obsjd", "size"),
    span_minutes=("obsjd", lambda jd: (jd.max() - jd.min()) * 24 * 60),
    seeing=("seeing", "median"),
    maglimit=("maglimit", "median"),
)
linkable = nights[nights["frames"] >= 4]
print(len(linkable), "nights with >= 4 good r-band frames")
linkable.sort_values("maglimit", ascending=False).head(12)

## 4. The split, decided **before** looking at any results
Written in `configs/acquire.yaml`, so it can't drift. The test night is the best one (deepest, sharpest)
and is kept for the very end.

In [ ]:
for split, split_nights in acquire["nights"].items():
    rows = nights.loc[split_nights]
    print(f"{split:10s} {len(split_nights):2d} nights   median seeing {rows['seeing'].median():.2f} px   median maglimit {rows['maglimit'].median():.2f}")

Note the test night is *better* than the training nights (deeper, sharper). That's a **distribution shift** we chose
on purpose: finding faint objects on the best night is the goal. We'll remember it when comparing scores.

## 5. Download (safe to re-run: existing files are skipped)

In [ ]:
reference_path = download_ztf_reference_image(acquire["field"], acquire["filter"], acquire["ccd"], acquire["quadrant"], raw_dir)
print("reference:", reference_path.name)

all_files = []
for split, split_nights in acquire["nights"].items():
    for night in split_nights:
        frames = select_night_exposures(metadata, night, acquire["filter"], acquire["frames_per_night"])
        files = download_ztf_images(frames, acquire["products"], raw_dir)
        all_files += files
        print(f"{split:10s} {night}: {len(frames)} frames, {len(files)} files")

## 6. Verify what we got

In [ ]:
total_bytes = sum(f.stat().st_size for f in all_files)
missing = [f for f in all_files if not f.exists()]
leftover_parts = list(raw_dir.rglob("*.part"))
print(f"{len(all_files)} files, {total_bytes / 1e9:.2f} GB, missing: {len(missing)}, unfinished .part files: {len(leftover_parts)}")

## 7. A first look at one training frame: science, difference, mask
The difference image is `science - reference`: stars cancel out, and **anything new or moving remains**.

In [ ]:
from asthunt.clean.read_fits_image import read_fits_image

first_train_night = acquire["nights"]["train"][0]
night_dir = raw_dir / first_train_night
science_path = sorted(night_dir.glob("*sciimg.fits"))[0]
science, header = read_fits_image(science_path)
difference, _ = read_fits_image(str(science_path).replace("sciimg.fits", "scimrefdiffimg.fits.fz"))
mask, _ = read_fits_image(str(science_path).replace("sciimg.fits", "mskimg.fits"))

cut = (slice(1200, 1500), slice(1200, 1500))      # a 300x300 pixel corner, so we can see details
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lo, hi = np.nanpercentile(science[cut], [5, 99])
axes[0].imshow(science[cut], origin="lower", cmap="gray", vmin=lo, vmax=hi); axes[0].set_title("science")
axes[1].imshow(difference[cut], origin="lower", cmap="gray", vmin=-30, vmax=30); axes[1].set_title("difference (sci - ref)")
axes[2].imshow(mask[cut], origin="lower", cmap="tab20"); axes[2].set_title("mask bits")
plt.show()
print("NaN fraction in difference image:", round(np.isnan(difference).mean(), 3))

## Exercises
1. Which night in `linkable` has the longest time span? Why might a *long* span make linking harder, not easier?
2. Pick a 2023-06 night: what's its median maglimit, and why did we not choose it?
3. Delete one downloaded `.fz` file and re-run section 5. What gets downloaded?